# NLP2025 Midterm Kaggle, ASAS Report

Pupipat Singkhorn, 6532142421

This is a overview of solutions, not a single run notebook. 

Because i have tried many different approaches, and this is a summary of the best ones. 
Some are lost in the process of trying different things. Some are other tools process e.g. Terminal cmd, excel. I will try to provide the most.
```bash
.
├── data
│   ├── demo
│   │   └── train.txt
│   ├── sample_submission.csv
│   ├── submission
│   │   ├── 4o-tfidf-wangchan-ag.csv
│   │   ├── chula-genie-plus.csv
│   │   ├── o1-zero-shot-tfidf.csv
│   │   ├── tfidf-wangchan-4sep-ag.csv
│   │   ├── tfidf-wangchan-mpl-esm.csv
│   │   ├── tfidf-wangchan-mpl-xgbr.csv
│   │   ├── tfidf-wangchan-xgbr-opt.csv
│   │   ├── tfidf-wangchan-xgbr-whtspc-lower.csv
│   │   ├── tfidf-wangchan-xgbr.csv
│   │   ├── wangchan-xgbr.csv
│   │   ├── whtspc-lower-tfidf-wangchan-ag-goodq.csv
│   │   └── whtspc-lower-tfidf-wangchan-ag-medq.csv
│   ├── test-TH.csv
│   ├── test.csv
│   ├── train-TH.csv
│   ├── train.csv
│   ├── translated_test.csv
│   ├── translated_train.csv
│   └── txt
│       ├── test-sort-by-set.txt
│       ├── test-without-set.txt
│       ├── test.txt
│       ├── train-sort-by-set.txt
│       ├── train-without-set.txt
│       └── train.txt
├── diagram.png
├── log.txt
├── notebooks
│   ├── 4o-tfidf-wangchan-ag.ipynb
│   ├── AutogluonModels
│   ├── format.ipynb
│   ├── gemini.ipynb
│   ├── main.ipynb
│   ├── o1-zero-shot-ag.ipynb
│   ├── o1-zero-shot-tfidf.ipynb
│   ├── submission.csv
│   ├── tfidf-wangchan-4sep-ag copy.ipynb
│   ├── tfidf-wangchan-4sep-ag.ipynb
│   ├── tfidf-wangchan-dimrdc-esm.ipynb
│   ├── tfidf-wangchan-mpl-esm.ipynb
│   ├── tfidf-wangchan-mpl-xgbr.ipynb
│   ├── tfidf-wangchan-xgbr-opt.ipynb
│   ├── tfidf-wangchan-xgbr-whtspc-lower.ipynb
│   ├── tfidf-wangchan-xgbr.ipynb
│   ├── translate.ipynb
│   ├── wangchan-xgbr.ipynb
│   ├── whtspc-lower-tfidf-wangchan-ag-goodq.ipynb
│   └── whtspc-lower-tfidf-wangchan-ag-medq.ipynb
└── overview.txt
```

## formatter

In [ ]:
# %%
# --- iPython Config --- #
from IPython import get_ipython
if 'IPython.extensions.autoreload' not in get_ipython().extension_manager.loaded:
    get_ipython().run_line_magic('load_ext', 'autoreload')
else:
    get_ipython().run_line_magic('reload_ext', 'autoreload')
%autoreload 2

# --- System and Path --- #
import os
import sys
REPO_PATH = os.path.abspath(os.path.join('..'))
if REPO_PATH not in sys.path:
    sys.path.append(REPO_PATH)
import warnings
warnings.filterwarnings("ignore")

# --- Data Manipulation --- #
import pandas as pd
import numpy as np

# %%
filename = 'train'
csv_data = pd.read_csv(os.path.join(REPO_PATH, 'data', f"{filename}.csv"))

# %%
text_data = ""
for _, row in csv_data.iterrows():
    text_data += f"id: {row['ID']}\n"
    text_data += f"set: {row['set']}\n"
    text_data += f"question: {row['question']}\n"
    text_data += f"answer: {row['answer']}\n"
    text_data += "score: \n\n"
text_data

# %%
# Save to plain text file
filepath = os.path.join(REPO_PATH, 'data', f"{filename}.txt")
with open(filepath, 'w', encoding='utf-8') as file:
    file.write(text_data)

## translate

In [ ]:
# --- iPython Config --- #
from IPython import get_ipython
if 'IPython.extensions.autoreload' not in get_ipython().extension_manager.loaded:
    get_ipython().run_line_magic('load_ext', 'autoreload')
else:
    get_ipython().run_line_magic('reload_ext', 'autoreload')
%autoreload 2

# --- System and Path --- #
import os
import sys
REPO_PATH = os.path.abspath(os.path.join('..'))
if REPO_PATH not in sys.path:
    sys.path.append(REPO_PATH)
import warnings
warnings.filterwarnings("ignore")

# --- Data Manipulation --- #
import pandas as pd
import numpy as np
import pandas as pd
from tqdm import tqdm
from deep_translator import GoogleTranslator
import os


# Load the CSV files

# Initialize the translator
translator = GoogleTranslator(source='en', target='th')

# Function to translate text with error handling
def translate_text(text):
    try:
        return translator.translate(text)
    except:
        return text  # Return original text if translation fails

# Function to translate a column with tqdm progress bar
def translate_column(column):
    return [translate_text(text) for text in tqdm(column, desc="Translating", unit="row")]

# Apply translation with progress tracking
train_df = pd.read_csv(os.path.join(REPO_PATH, "data/train.csv"))
train_df["question"] = translate_column(train_df["question"])
train_df["answer"] = translate_column(train_df["answer"])
train_df.to_csv(os.path.join(REPO_PATH, "translated_train.csv"), index=False)
del train_df

test_df = pd.read_csv(os.path.join(REPO_PATH, "data/test.csv"))
test_df["question"] = translate_column(test_df["question"])
test_df["answer"] = translate_column(test_df["answer"])
test_df.to_csv(os.path.join(REPO_PATH, "translated_test.csv"), index=False)
del test_df
print("Translation completed! Translated files are saved.")

## tfidf-wangchan-4sep-ag
this is only one best solution of the many as provided above.

In [1]:
# --- iPython Config --- #
from IPython import get_ipython
if 'IPython.extensions.autoreload' not in get_ipython().extension_manager.loaded:
    get_ipython().run_line_magic('load_ext', 'autoreload')
else:
    get_ipython().run_line_magic('reload_ext', 'autoreload')
%autoreload 2

# --- System and Path --- #
import os
import sys
REPO_PATH = os.path.abspath(os.path.join('..'))
if REPO_PATH not in sys.path:
    sys.path.append(REPO_PATH)
import warnings
warnings.filterwarnings("ignore")

# --- Data Manipulation --- #
import pandas as pd
import numpy as np

In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from transformers import AutoTokenizer, AutoModel
import torch
from autogluon.tabular import TabularPredictor

# Helper functions defined in your original script
def clean_text(text):
    return " ".join(text.lower().split())

def combine_text(question, answer):
    q = question if isinstance(question, str) else ""
    a = answer if isinstance(answer, str) else ""
    return clean_text(q + " " + a)

class ThaiBERTEmbedder:
    def __init__(self, model_name="airesearch/wangchanberta-base-att-spm-uncased", device=None):
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModel.from_pretrained(model_name).to(self.device)
        self.model.eval()

    def encode(self, text_list, batch_size=16, max_length=128):
        embeddings = []
        for i in range(0, len(text_list), batch_size):
            batch_text = text_list[i : i + batch_size]
            inputs = self.tokenizer(batch_text, padding=True, truncation=True,
                                    max_length=max_length, return_tensors="pt").to(self.device)
            with torch.no_grad():
                outputs = self.model(**inputs)
            embeddings.append(outputs.last_hidden_state[:, 0, :].cpu().numpy())
        return np.vstack(embeddings)

# Load data
train_df = pd.read_csv(os.path.join(REPO_PATH, "data","train.csv"))
test_df = pd.read_csv(os.path.join(REPO_PATH, "data","test.csv"))
submission_df = pd.read_csv(os.path.join(REPO_PATH, "data","sample_submission.csv"))

# Initialize TF-IDF and BERT
vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=10000)
bert_embedder = ThaiBERTEmbedder()

final_predictions = []

# Train and predict separately for each Q set
for q_set in ["Q1", "Q2", "Q3", "Q4"]:
    print(f"Processing {q_set}...")

    train_subset = train_df[train_df["set"] == q_set]
    test_subset = test_df[test_df["set"] == q_set]

    train_texts = train_subset.apply(lambda x: combine_text(x["question"], x["answer"]), axis=1)
    test_texts = test_subset.apply(lambda x: combine_text(x["question"], x["answer"]), axis=1)

    X_tfidf_train = vectorizer.fit_transform(train_texts).toarray()
    X_tfidf_test = vectorizer.transform(test_texts).toarray()

    X_bert_train = bert_embedder.encode(train_texts.tolist())
    X_bert_test = bert_embedder.encode(test_texts.tolist())

    X_train = np.hstack([X_tfidf_train, X_bert_train])
    X_test = np.hstack([X_tfidf_test, X_bert_test])

    y_train = train_subset["score"].values

    train_data = pd.DataFrame(X_train, columns=[f"f{i}" for i in range(X_train.shape[1])])
    train_data["score"] = y_train

    predictor = TabularPredictor(label="score", problem_type="regression").fit(train_data)

    test_data = pd.DataFrame(X_test, columns=[f"f{i}" for i in range(X_test.shape[1])])
    preds = predictor.predict(test_data)

    preds_df = pd.DataFrame({"ID": test_subset["ID"].values, "score": preds})
    final_predictions.append(preds_df)

# Merge predictions from all Q sets
final_submission = pd.concat(final_predictions).set_index("ID").reindex(submission_df["ID"]).reset_index()

# Ensure no null values in submission
assert final_submission.isnull().sum().sum() == 0, "Null values detected!"

# Save submission
final_submission.to_csv("submission.csv", index=False)
print("Final submission file saved as 'submission.csv'")


Processing Q1...


No path specified. Models will be saved in: "AutogluonModels/ag-20250310_075405"
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.2
Python Version:     3.11.10
Operating System:   Darwin
Platform Machine:   arm64
Platform Version:   Darwin Kernel Version 24.3.0: Thu Jan  2 20:23:36 PST 2025; root:xnu-11215.81.4~3/RELEASE_ARM64_T8112
CPU Count:          8
Memory Avail:       1.61 GB / 8.00 GB (20.1%)
Disk Space Avail:   23.75 GB / 228.27 GB (10.4%)
No presets specified! To achieve strong results with AutoGluon, it is recommended to use the available presets. Defaulting to `'medium'`...
	Recommended Presets (For more details refer to https://auto.gluon.ai/stable/tutorials/tabular/tabular-essentials.html#presets):
	presets='experimental' : New in v1.2: Pre-trained foundation model + parallel fits. The absolute best accuracy without consideration for inference speed. Does not support GPU.
	presets='best'         : Maximize accuracy. 

Processing Q2...


No path specified. Models will be saved in: "AutogluonModels/ag-20250310_075424"
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.2
Python Version:     3.11.10
Operating System:   Darwin
Platform Machine:   arm64
Platform Version:   Darwin Kernel Version 24.3.0: Thu Jan  2 20:23:36 PST 2025; root:xnu-11215.81.4~3/RELEASE_ARM64_T8112
CPU Count:          8
Memory Avail:       1.49 GB / 8.00 GB (18.6%)
Disk Space Avail:   23.74 GB / 228.27 GB (10.4%)
No presets specified! To achieve strong results with AutoGluon, it is recommended to use the available presets. Defaulting to `'medium'`...
	Recommended Presets (For more details refer to https://auto.gluon.ai/stable/tutorials/tabular/tabular-essentials.html#presets):
	presets='experimental' : New in v1.2: Pre-trained foundation model + parallel fits. The absolute best accuracy without consideration for inference speed. Does not support GPU.
	presets='best'         : Maximize accuracy. 

Processing Q3...


...

Processing Q4...


No path specified. Models will be saved in: "AutogluonModels/ag-20250310_075459"
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.2
Python Version:     3.11.10
Operating System:   Darwin
Platform Machine:   arm64
Platform Version:   Darwin Kernel Version 24.3.0: Thu Jan  2 20:23:36 PST 2025; root:xnu-11215.81.4~3/RELEASE_ARM64_T8112
CPU Count:          8
Memory Avail:       1.60 GB / 8.00 GB (20.0%)
Disk Space Avail:   23.72 GB / 228.27 GB (10.4%)
No presets specified! To achieve strong results with AutoGluon, it is recommended to use the available presets. Defaulting to `'medium'`...
	Recommended Presets (For more details refer to https://auto.gluon.ai/stable/tutorials/tabular/tabular-essentials.html#presets):
	presets='experimental' : New in v1.2: Pre-trained foundation model + parallel fits. The absolute best accuracy without consideration for inference speed. Does not support GPU.
	presets='best'         : Maximize accuracy. 

[1000]	valid_set's rmse: 1.15669
[2000]	valid_set's rmse: 1.15668
[3000]	valid_set's rmse: 1.15668
[4000]	valid_set's rmse: 1.15668
[5000]	valid_set's rmse: 1.15668
[6000]	valid_set's rmse: 1.15668
[7000]	valid_set's rmse: 1.15668
[8000]	valid_set's rmse: 1.15668
[9000]	valid_set's rmse: 1.15668
[10000]	valid_set's rmse: 1.15668


	-1.1567	 = Validation score   (-root_mean_squared_error)
	9.4s	 = Training   runtime
	0.01s	 = Validation runtime
Fitting model: WeightedEnsemble_L2 ...
	Ensemble Weights: {'NeuralNetFastAI': 0.929, 'KNeighborsDist': 0.071}
	-0.8426	 = Validation score   (-root_mean_squared_error)
	0.02s	 = Training   runtime
	0.0s	 = Validation runtime
AutoGluon training complete, total runtime = 15.98s ... Best model: WeightedEnsemble_L2 | Estimated inference throughput: 2265.1 rows/s (23 batch size)
TabularPredictor saved. To load, use: predictor = TabularPredictor.load("/Users/pupipatsingkhorn/Developer/repositories/NLP/nlp-2025-midterm-kaggle-asas/notebooks/AutogluonModels/ag-20250310_075459")


Final submission file saved as 'final_submission.csv'
